# Orislop Temporal MoE v5 — T4 retraining

This freezes the four trained temporal experts, caches their outputs once, retrains Stage 2 fusion/router, calibrates Stage 3 on validation, evaluates once on test, and packages the result. Spatial stays independent.

Before running: select **Runtime → Change runtime type → T4 GPU** and add an `HF_TOKEN` secret with write access. A full run downloads about 59 GiB and needs roughly 68 GiB of free local storage.

In [ ]:
import subprocess
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu)
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
from pathlib import Path
import os, subprocess, zipfile
REPO = Path('/content/Orislop-landing')
UPLOAD_BUNDLE = Path('/content/orislop-temporal-colab-upload.zip')
if UPLOAD_BUNDLE.is_file():
    REPO.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(UPLOAD_BUNDLE) as archive:
        archive.extractall(REPO)
    print('Loaded the manually uploaded Orislop training bundle.')
elif not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/coolguy860/Orislop-landing.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', str(REPO / 'training/orislop_temporal_retrain/requirements-colab.txt')], check=True)
os.chdir(REPO)
print('Repository ready at', REPO)


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('HF_TOKEN')
assert token, 'Add HF_TOKEN under the key icon in the Colab sidebar.'
os.environ['HF_TOKEN'] = token
print('Drive mounted and Hugging Face token loaded (token not displayed).')


## Choose smoke or full training

Use `SMOKE_SHARDS = 1` for the first wiring check. Change it to `0` for the real training run. Smoke checkpoints are not production evidence.

In [ ]:
import json
SMOKE_SHARDS = 1  # Change to 0 for the full ~59 GiB dataset.
OUTPUT_SUBDIR = 't4_temporal_fusion_v5_smoke' if SMOKE_SHARDS else 't4_temporal_fusion_v5'
RUN_ROOT = f'/content/drive/MyDrive/orislop-temporal-v5/{OUTPUT_SUBDIR}'
prefix = 'hf:gonnerthetooner/deepfake-temporal-moe/a100_high_vram_60gb_v1'
config = {
    'run_root': RUN_ROOT,
    'expert_cache_root': f'{RUN_ROOT}/expert-cache',
    'data': {
        'local_root': '/content/orislop-temporal-data',
        'dataset_repo': 'gonnerthetooner/deepfake-frame-views-balanced-fusion-v1',
        'revision': 'main',
        'smoke_shards': SMOKE_SHARDS,
    },
    'model_repo': 'gonnerthetooner/deepfake-temporal-moe',
    'output_subdir': OUTPUT_SUBDIR,
    'hf_private': False,
    'upload_bundle': True,
    'experts': {name: f'{prefix}/stage1_{name}_best.pt' for name in ('micro', 'mid', 'long', 'extra_long')},
    'training': {
        'precision': 'fp16', 'embedding_dim': 256, 'fusion_dim': 256,
        'fusion_layers': 4, 'fusion_heads': 4, 'fusion_dropout': 0.1,
        'fusion_lr': 1e-4, 'fusion_weight_decay': 1e-4, 'fusion_epochs': 1 if SMOKE_SHARDS else 20,
        'early_stopping_patience': 4, 'early_stopping_min_delta': 5e-4, 'expert_dropout': 0.1,
        'cached_fusion_batch_size': 256, 'cache_shard_records': 1024,
        'batch_size': 1, 'grad_accum_steps': 4, 'clip_frame_chunk_size': 2, 'num_workers': 0, 'seed': 1337,
        'minimum_train_per_class': 1000, 'minimum_val_per_class': 100, 'minimum_test_per_class': 100,
    },
    'gates': {
        'threshold_method': 'target_real_fpr', 'maximum_genuine_hide_rate': 0.001,
        'minimum_recall': 0.90, 'maximum_ece': 0.03, 'minimum_test_records': 800,
    },
    'av': {'enabled': False, 'lip_checkpoint': ''},
}
CONFIG_PATH = Path('/content/temporal_fusion_v5_config.json')
CONFIG_PATH.write_text(json.dumps(config, indent=2) + '\n')
print('Mode:', 'SMOKE' if SMOKE_SHARDS else 'FULL')
print('Persistent run folder:', RUN_ROOT)


In [ ]:
subprocess.run(['python', 'training/orislop_temporal_retrain/colab_retrain.py', 'self-test'], check=True)
print('Local orchestrator contract passed.')


In [ ]:
# Safe to rerun: checkpoints and the expert-output cache resume from Drive.
subprocess.run([
    'python', 'training/orislop_temporal_retrain/colab_retrain.py', 'run',
    '--config', str(CONFIG_PATH),
], check=True)


In [ ]:
result_path = Path(RUN_ROOT) / 'pipeline_result.json'
result = json.loads(result_path.read_text())
print(json.dumps(result['release'], indent=2))
print('Bundle folder:', result['bundle'])
print('ZIP archive:', result['archive'])
assert result['release']['productionEligible'] is False


## Resume after a disconnect

If the expert cache is already complete, replace the pipeline cell with:

```python
subprocess.run(['python', 'training/orislop_temporal_retrain/colab_retrain.py', 'run', '--config', str(CONFIG_PATH), '--start-at', 'train'], check=True)
```

Do not promote the model just because offline metrics pass. The bundle remains shadow-only pending grouped-data sign-off and 10,000 reviewed shadow decisions.